In [1]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

In [1]:
# use your `key` and `endpoint` environment variables
key = <>
endpoint = "https://azure-ocr-giridhar.services.ai.azure.com/"


SyntaxError: invalid syntax (1294025506.py, line 2)

In [10]:

client = DocumentIntelligenceClient(
        endpoint=endpoint, credential=AzureKeyCredential(key)
    )

In [11]:
# Load a PDF file into byte stream
with open("../data/inputs/arxiv/pdfs/2303.13740.pdf", 'rb') as pdf_file:
    pdf_data = pdf_file.read()


In [12]:
def analyze_read(byte_stream):
    poller = client.begin_analyze_document(
        "prebuilt-read", AnalyzeDocumentRequest(bytes_source=byte_stream)
    )
    result: AnalyzeResult = poller.result()
    return result

In [13]:
response = analyze_read(pdf_data)

ResourceNotFoundError: (404) Resource not found
Code: 404
Message: Resource not found

In [ ]:

# helper functions
def get_words(page, line):
    result = []
    for word in page.words:
        if _in_span(word, line.spans):
            result.append(word)
    return result


def _in_span(word, spans):
    for span in spans:
        if word.span.offset >= span.offset and (word.span.offset + word.span.length) <= (span.offset + span.length):
            return True
    return False




    print("----Languages detected in the document----")
    if result.languages is not None:
        for language in result.languages:
            print(f"Language code: '{language.locale}' with confidence {language.confidence}")

    print("----Styles detected in the document----")
    if result.styles:
        for style in result.styles:
            if style.is_handwritten:
                print("Found the following handwritten content: ")
                print(",".join([result.content[span.offset : span.offset + span.length] for span in style.spans]))
            if style.font_style:
                print(f"The document contains '{style.font_style}' font style, applied to the following text: ")
                print(",".join([result.content[span.offset : span.offset + span.length] for span in style.spans]))

    for page in result.pages:
        print(f"----Analyzing document from page #{page.page_number}----")
        print(f"Page has width: {page.width} and height: {page.height}, measured with unit: {page.unit}")

        if page.lines:
            for line_idx, line in enumerate(page.lines):
                words = get_words(page, line)
                print(
                    f"...Line # {line_idx} has {len(words)} words and text '{line.content}' within bounding polygon '{line.polygon}'"
                )

                for word in words:
                    print(f"......Word '{word.content}' has a confidence of {word.confidence}")

        if page.selection_marks:
            for selection_mark in page.selection_marks:
                print(
                    f"...Selection mark is '{selection_mark.state}' within bounding polygon "
                    f"'{selection_mark.polygon}' and has a confidence of {selection_mark.confidence}"
                )

    if result.paragraphs:
        print(f"----Detected #{len(result.paragraphs)} paragraphs in the document----")
        for paragraph in result.paragraphs:
            print(f"Found paragraph with role: '{paragraph.role}' within {paragraph.bounding_regions} bounding region")
            print(f"...with content: '{paragraph.content}'")

        result.paragraphs.sort(key=lambda p: (p.spans.sort(key=lambda s: s.offset), p.spans[0].offset))
        print("-----Print sorted paragraphs-----")
        for idx, paragraph in enumerate(result.paragraphs):
            print(
                f"...paragraph:{idx} with offset: {paragraph.spans[0].offset} and length: {paragraph.spans[0].length}"
            )

    print("----------------------------------------")


if __name__ == "__main__":
    analyze_read()

In [5]:
from PIL import Image
import pypdfium2 as pdfium
from pathlib import Path

In [6]:
# Local PDF path
pdf_path = Path("../data/inputs/arxiv/pdfs/2303.13740.pdf")

# Render all pages to base64 PNGs
pdf = pdfium.PdfDocument(str(pdf_path))
page_images_b64 = []
for i in range(len(pdf)):
    page = pdf[i]
    # Render at 200 DPI (scale factor = 200/72 ≈ 2.77)
    page_images_b64.append(page.render(scale=2.77).to_pil())

In [ ]:
import re


from transformers import NougatProcessor, VisionEncoderDecoderModel
from datasets import load_dataset
import torch


processor = NougatProcessor.from_pretrained("facebook/nougat-base")
model = VisionEncoderDecoderModel.from_pretrained("facebook/nougat-base")



The image processor of type `NougatImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 640/640 [00:00<00:00, 1364.86it/s, Materializing param=encoder.encoder.layers.3.blocks.1.output.dense.weight]                         


NameError: name 'Accelerator' is not defined

In [8]:
from accelerate import Accelerator

In [ ]:
page_images_b64[0].

TypeError: 'Image' object is not subscriptable

In [12]:
type(page_images_b64[0])

PIL.Image.Image

In [17]:
processor?

Signature:     
processor(
    images=None,
    text=None,
    do_crop_margin: bool | None = None,
    do_resize: bool | None = None,
    size: dict[str, int] | None = None,
    resample: 'PILImageResampling' = None,
    do_thumbnail: bool | None = None,
    do_align_long_axis: bool | None = None,
    do_pad: bool | None = None,
    do_rescale: bool | None = None,
    rescale_factor: int | float | None = None,
    do_normalize: bool | None = None,
    image_mean: float | list[float] | None = None,
    image_std: float | list[float] | None = None,
    data_format: Optional[ForwardRef('ChannelDimension')] = 'channels_first',
    input_data_format: Union[str, ForwardRef('ChannelDimension'), NoneType] = None,
    text_pair: str | list[str] | list[list[str]] | None = None,
    text_target: str | list[str] | list[list[str]] | None = None,
    text_pair_target: str | list[str] | list[list[str]] | None = None,
    add_special_tokens: bool = True,
    padding: bool | str | transformers.utils.ge

In [22]:
pixel_values = processor(page_images_b64[0], return_tensors="pt",do_crop_margin=True, do_resize=False, do_thumbnail=False,do_align_long_axis=False).pixel_values

In [23]:
pixel_values

tensor([[[[255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          ...,
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255]],

         [[255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          ...,
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255]],

         [[255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          ...,
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255]]]], dtype=torch.uint8)

In [24]:
device = Accelerator().device
model.to(device)
# prepare PDF image for the model


# generate transcription (here we only generate 30 tokens)
outputs = model.generate(
    pixel_values.to(device),
    min_length=1,
    max_new_tokens=3000,
)

sequence = processor.batch_decode(outputs, skip_special_tokens=True)[0]
if hasattr(processor, "post_process_generation"):
    sequence = processor.post_process_generation(sequence, fix_markdown=False)
else:
    # Some tokenizers don't implement post_process_generation
    sequence = sequence
# note: we're using repr here such for the sake of printing the \n characters, feel free to just print the sequence
print(repr(sequence))

RuntimeError: Input type (unsigned char) and bias type (float) should be the same

In [2]:
import os
import json

In [4]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 26.4 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 29.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]dateutil]


In [3]:
arxiv_files_data = list()
with open('../data/inputs/arxiv/arxiv_samples.jsonl','r') as file:
    for line in file.readlines():
        arxiv_files_data.append(json.loads(line))


In [5]:
import pandas as pd

In [6]:
df = pd.DataFrame(arxiv_files_data)

In [8]:
df['primary_subject'].value_counts()

primary_subject
Earth and Planetary Astrophysics (astro-ph.EP)                110
Astrophysics of Galaxies (astro-ph.GA)                        110
Solar and Stellar Astrophysics (astro-ph.SR)                  110
High Energy Astrophysical Phenomena (astro-ph.HE)             110
Instrumentation and Methods for Astrophysics (astro-ph.IM)    110
                                                             ... 
Applications (stat.AP)                                        110
Other Statistics (stat.OT)                                    110
Methodology (stat.ME)                                         110
Machine Learning (stat.ML)                                    110
Computation (stat.CO)                                         110
Name: count, Length: 148, dtype: int64

In [11]:
df.groupby('primary_subject').sample(10)['arxiv_id'].values.tolist()

['0802.2667',
 '0712.3399',
 '0802.0504',
 '0802.0154',
 '0802.0237',
 '0804.1764',
 '0806.2893',
 '0707.0949',
 '0802.4023',
 '0809.4997',
 '0706.0440',
 '0708.1191',
 '0904.1144',
 '0808.0172',
 '0904.1456',
 '0804.3676',
 '0807.3416',
 '0811.3689',
 '0704.2533',
 '1001.1698',
 '0704.3386',
 '0705.1012',
 '0704.2858',
 '0704.2578',
 '0704.3627',
 '0705.0070',
 '0705.0659',
 '0705.0603',
 '0704.3912',
 '0705.0302',
 '0708.4262',
 '0708.3036',
 '0707.3103',
 '0708.1581',
 '0802.4357',
 '0708.2995',
 '0704.4002',
 '0709.3489',
 '0802.3652',
 '0708.0659',
 '0704.0337',
 '0704.0687',
 '0707.0045',
 '0704.2775',
 '0704.0347',
 '0704.2360',
 '0707.1445',
 '0705.3659',
 '0704.0497',
 '0707.1952',
 '0706.1401',
 '0705.0304',
 '0709.0421',
 '0709.0427',
 '0806.1894',
 '0710.5805',
 '0711.1369',
 '0803.3675',
 '0803.4055',
 '0804.0099',
 '1706.07283',
 '1706.05262',
 '1706.08103',
 '1706.04777',
 '1706.06222',
 '1706.04596',
 '1706.07538',
 '1704.06414',
 '1704.07918',
 '1704.06599',
 '0708.050